# Bernasconi-Krylov Colab Environment Setup & Build

This notebook configures Google Colab (A100 GPU), installs cuQuantum & Eigen3, extracts `colab.zip`, and compiles the `run_benchmark` binary.

### Instructions:
1. In the Colab file browser (left sidebar), upload `colab.zip` to `/content/`.
2. Run **Cell 1** to set up the environment and build the binary.
3. Run **Cell 2** (or your own command) to launch simulations.

In [ ]:
# ==============================================================================
# Step 1 & 2: Install dependencies & Locate cuQuantum
# ==============================================================================
import os, sys, glob, shutil

!nvidia-smi
!pip install -q cuquantum-cu12
!apt-get install -y -qq libeigen3-dev

so_candidates = glob.glob("/usr/local/lib/python*/dist-packages/cuquantum/lib/libcustatevec.so*")
if not so_candidates:
    so_candidates = glob.glob("/usr/local/lib/python*/dist-packages/**/libcustatevec.so*", recursive=True)
if not so_candidates:
    raise FileNotFoundError("Could not find libcustatevec.so. Please verify cuquantum-cu12 installation.")

cuquantum_so = sorted(so_candidates, key=lambda x: len(x))[0]
cuquantum_lib_dir = os.path.dirname(cuquantum_so)

inc_candidates = glob.glob("/usr/local/lib/python*/dist-packages/cuquantum/include")
if not inc_candidates:
    inc_candidates = glob.glob("/usr/local/lib/python*/dist-packages/**/custatevec.h", recursive=True)
    cuquantum_include = os.path.dirname(inc_candidates[0]) if inc_candidates else None
else:
    cuquantum_include = inc_candidates[0]

cuquantum_root = os.path.dirname(cuquantum_lib_dir)

os.environ["CUQUANTUM_ROOT"] = cuquantum_root
os.environ["LD_LIBRARY_PATH"] = f"{cuquantum_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"

print(f"[+] cuQuantum Root:    {cuquantum_root}")
print(f"[+] cuStateVec Lib:    {cuquantum_so}")
print(f"[+] cuQuantum Include: {cuquantum_include}")

# ==============================================================================
# Step 3: Unpack project archive
# ==============================================================================
archive_candidates = ["/content/colab.zip", "/colab.zip", "colab.zip"]
archive = next((a for a in archive_candidates if os.path.exists(a)), None)
if archive:
    print(f"[+] Extracting {archive} to /content/...")
    !unzip -o -q {archive} -d /content/
else:
    print("[!] Warning: colab.zip not found in /content/ or /. Please upload colab.zip.")

%cd /content/

# ==============================================================================
# Step 4: Build run_benchmark with CMake
# ==============================================================================
!cmake -B build -S . \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES="native" \
    -DCUQUANTUM_ROOT="{cuquantum_root}" \
    -DCUQUANTUM_INCLUDE_DIR="{cuquantum_include}" \
    -DCUSTATEVEC_LIB="{cuquantum_so}"

!cmake --build build --target run_benchmark -j$(nproc)

print("\n[✓] Build complete! Binary ready at /content/build/run_benchmark")

In [ ]:
# Example Execution (Run when ready):
# !./build/run_benchmark configs/probability_scan_n28.cfg
# !python3 scripts/plot_probability_dynamics.py data_processed/probability_dynamics_n28.csv figures/n28